# ResNet18 on CIFAR-10

Data loaders, model setup, and a training loop instrumented for **memory** and
**throughput** profiling


In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

from convolutions.models.resnet import ResNet18, ResNetInitParams
from convolutions.train import train


In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"using device: {device}")


using device: mps


In [3]:
batch_size = 128
data_root = "../../data"

# CIFAR-10 is 3x32x32 RGB. Standard per-channel mean/std for CIFAR-10.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# Train-time light augmentation; test-time just tensor + normalize.
train_transform = v2.Compose([
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = datasets.CIFAR10(
    root=data_root, train=True, download=True, transform=train_transform
)
test_dataset = datasets.CIFAR10(
    root=data_root, train=False, download=True, transform=test_transform
)

# num_workers>0 parallelises the (augmentation) pipeline so the GPU is not
# starved; keeps throughput measurements honest. persistent_workers avoids
# re-spawning between epochs.
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=2, persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=2, persistent_workers=True,
)

print(f"train: {len(train_dataset)} samples | test: {len(test_dataset)} samples")


train: 50000 samples | test: 10000 samples


In [4]:
# sanity check
X_batch, y_batch = next(iter(train_loader))
print("batch X shape:", X_batch.shape)  # [batch_size, 3, 32, 32]
print("batch y shape:", y_batch.shape)  # [batch_size]

batch X shape: torch.Size([128, 3, 32, 32])
batch y shape: torch.Size([128])


In [5]:
# ResNet18: 4 stages of 2 residual layers each -> (2,64),(2,128),(2,256),(2,512)
model = ResNet18(ResNetInitParams(
    arch=[(2, 64), (2, 128), (2, 256), (2, 512)],
    num_classes=10,
))
model.apply_init((1, 3, 32, 32))

model = model.to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params/1e6:.2f}M")


parameters: 11.18M


In [6]:
# train() (in train.py) reports per-epoch throughput (img/s) and MPS memory
# (live / peak / driver) alongside the loss.
epochs = 100
loss_fn = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(
    model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4
)
# Cosine-decay the lr from 0.1 down to ~0 over the full run.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

train(model, loss_fn, optimiser, train_loader, epochs=epochs, device=device, scheduler=scheduler)


>>> epoch : 0 | epoch loss (train) : 1.556325582896962 | lr 0.0100 | 59.2s | 844.1 img/s | mem live 170.3MB peak 196.7MB driver 1284.6MB >>>
>>> epoch : 1 | epoch loss (train) : 1.0752501711821008 | lr 0.0100 | 62.5s | 800.1 img/s | mem live 170.4MB peak 173.5MB driver 1284.6MB >>>
>>> epoch : 2 | epoch loss (train) : 0.8905326285020775 | lr 0.0100 | 61.4s | 814.8 img/s | mem live 170.3MB peak 173.5MB driver 1284.6MB >>>
>>> epoch : 3 | epoch loss (train) : 0.7521240490171915 | lr 0.0100 | 60.4s | 828.3 img/s | mem live 170.4MB peak 173.5MB driver 1284.6MB >>>
>>> epoch : 4 | epoch loss (train) : 0.6633302510699348 | lr 0.0100 | 60.4s | 827.5 img/s | mem live 170.3MB peak 173.5MB driver 1284.6MB >>>
>>> epoch : 5 | epoch loss (train) : 0.5854628579238491 | lr 0.0099 | 60.0s | 833.4 img/s | mem live 170.4MB peak 173.5MB driver 1284.6MB >>>
>>> epoch : 6 | epoch loss (train) : 0.5311786434077241 | lr 0.0099 | 60.1s | 831.3 img/s | mem live 170.3MB peak 173.5MB driver 1284.6MB >>>
>>> epo

In [7]:
# Evaluate on the full CIFAR-10 test set: top-1 and top-5 accuracy
model.eval()
top1 = 0
top5 = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X, y = X_batch.to(device), y_batch.to(device)
        # top-5 indices per sample, ordered most-confident first
        _, top5_idx = model(X).topk(5, dim=1)
        correct = top5_idx == y.unsqueeze(1)  # [batch, 5] bool
        top1 += correct[:, 0].sum().item()
        top5 += correct.any(dim=1).sum().item()
        total += y.size(0)

print(f"{'metric':<8} {'accuracy':>10} {'correct':>12}")
print(f"{'-' * 32}")
print(f"{'top-1':<8} {top1 / total:>9.2%} {f'{top1}/{total}':>12}")
print(f"{'top-5':<8} {top5 / total:>9.2%} {f'{top5}/{total}':>12}")


metric     accuracy      correct
--------------------------------
top-1       92.97%   9297/10000
top-5       99.71%   9971/10000
